# Chapter 6: The Geometric Judge — Scoring Beyond Binary Correctness

**Learning objectives:**
- Understand how GMSJudge produces a GeometricVerdict with four continuous signals
- Interpret geodesic confidence, tension score and holonomy score
- See how the composite overall confidence is computed
- Run the GMS baseline and confirm 100% accuracy

> **LLM API key required.** This notebook makes real LLM calls
> (e.g. `anthropic.Anthropic()`). The provider SDK reads the key
> from the process environment, not from `.env` — so either
> `source .env` before launching Jupyter, or export
> `ANTHROPIC_API_KEY` (or whichever provider key matches your
> `GMS_LLM_MODEL`) in the shell that runs the kernel.
>
> See `.env.example` at the repo root for the full list of
> supported provider env vars, and `docs/internal_team_install.md`
> §3B › "LLM API key setup" for the step-by-step recipe.


---
## Preamble

Chapter 5 generated test questions with provable ground truth. This chapter shows how the **GMSJudge** scores agent answers against that ground truth — not with a binary pass/fail, but with four continuous geometric signals that capture *how right* or *how wrong* an answer is.

Key modules:
- `harness.doe.judge` — `GMSJudge`, `GeometricVerdict`, `create_judge`
- `docgms.verify` — `check_contradictions`, `check_holonomy`
- `docgms.extract` — `ExtractedTriple`, `ScoredTriple`

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and create the judge
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig, create_judge

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

judge = create_judge(store)
print(f"\nJudge created with weights: {judge.weights}")

---

## Why Binary Scoring Is Not Enough

Traditional evaluation scores an LLM answer as correct or incorrect by string matching against the ground truth. Consider this question:

> *What is the Test AUC of MDL-001?*
> Ground truth: **0.8412**

Three agent answers:
1. `"0.8412"` — **exact match** (correct)
2. `"0.84"` — **truncated** (wrong by string match, but the agent clearly knows the value)
3. `"0.95"` — **hallucinated** (wrong, and not even close)

Binary scoring treats answers 2 and 3 identically: both get a score of 0. The geometric judge distinguishes between them by measuring *how well the answer fits the manifold*.

---

## The GeometricVerdict

The `GMSJudge.judge()` method returns a `GeometricVerdict` with four continuous signals:

| Signal | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `geodesic_confidence` | $\exp(-\bar{d})$ | $[0, 1]$ | How well the answer fits the manifold |
| `tension_score` | $\bar{E}$ | $[0, 2]$ | Mean tension energy (0 = agreement) |
| `holonomy_score` | $\bar{\delta}$ | $[0, \infty)$ | Mean holonomy defect (0 = consistent) |
| `overall_confidence` | weighted sum | $[0, 1]$ | Composite score |

### What each signal means geometrically

**Geodesic confidence** measures how well a ground truth triple fits the learned manifold. Each relation $r$ is a rotation matrix $R_r \in \mathrm{SO}(m)$ on the unit hypersphere. Given a triple $(h, r, t)$, the GMS computes the geodesic distance $d = \arccos \langle R_r \tilde{v}_h, \tilde{v}_t \rangle$ between the rotated head and the tail embedding. Confidence $= \exp(-\bar{d})$:
- $d = 0$ (0°): rotated head lands exactly on the tail → confidence = 1.0
- $d = \pi/2$ (~1.57 rad, 90°): orthogonal → confidence ≈ 0.21
- $d = \pi$ (180°): antipodal → confidence ≈ 0.04

Intuitively: *"Does the manifold think this triple belongs?"* A well-learned triple sits tightly on the surface; a rare relation may be embedded more loosely.

**Tension score** measures logical consistency between entity pairs via their polarity embeddings ($u$-space). The tension energy $E(a,b) = 2\sin(\theta/2)$ where $\theta$ is the angle between polarity vectors:
- $E \approx 0$ (θ ≈ 0°): logically aligned (**entailment**, threshold $E < 0.8$)
- $E \approx \sqrt{2} \approx 1.41$ (θ = 90°): **unrelated** ($0.8 \le E \le 1.6$)
- $E \approx 2$ (θ = 180°): logically opposed (**contradiction**, $E > 1.6$)

The "unrelated" middle range is common and healthy — it means neighboring entities serve different roles (e.g. a numeric metric and a risk rating).

**Holonomy score** captures path consistency in multi-hop reasoning. When a question traverses two or more relations ($h \xrightarrow{r_1} m \xrightarrow{r_2} t$), the judge composes the relation rotors $R_{r_2} R_{r_1}$ and compares with the direct rotor $R_r$. The defect $\delta = \| R_{\text{path}} R_r^{-1} - I \|_F$ measures how much the composed path "twists" compared to the direct route — like parallel transport around a curved surface. For single-hop questions (one triple, no path to compose), holonomy is zero by definition.

### Overall confidence

$$\text{overall} = w_{\text{exact}} \cdot \mathbf{1}_{\text{exact}} + w_{\text{geo}} \cdot \text{geodesic\_conf} + w_{\text{tens}} \cdot (1 - \bar{E}/2) + w_{\text{hol}} \cdot \exp(-\bar{\delta})$$

Each component is normalized to $[0, 1]$ before weighting. Tension $\bar{E} \in [0,2]$ is mapped to quality $(1 - \bar{E}/2)$ so low tension → high quality. Holonomy $\bar{\delta}$ is mapped via $\exp(-\bar{\delta})$ so zero defect → quality 1.0.

The five-step judge pipeline:
1. **Extract triples** from the LLM answer and question metadata
2. **Score triples** against the manifold via `score_triple()`
3. **Check contradictions** via `tension_energy()` on entity pairs
4. **Check holonomy** via rotor composition on multi-hop paths
5. **Aggregate** into the weighted overall confidence

---

## Lab 6.1 — Judge Agent Answers

Let us judge three answers to the same question: exact correct, approximate and hallucinated.

In [ ]:
# Lab 6.1: Judge three answer scenarios

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

# Scenario 1: Exact correct
v1 = judge.judge(
    llm_answer="0.8412",
    ground_truth="0.8412",
    exact_correct=True,
    exact_detail="Exact match",
    question_metadata=meta,
)

# Scenario 2: Approximate (truncated)
v2 = judge.judge(
    llm_answer="0.84",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.84",
    question_metadata=meta,
)

# Scenario 3: Hallucinated
v3 = judge.judge(
    llm_answer="0.95",
    ground_truth="0.8412",
    exact_correct=False,
    exact_detail="Expected 0.8412, got 0.95",
    question_metadata=meta,
)

# Summary table
print(f"{'Scenario':18s} {'Correct':>8s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s}")
print("-" * 68)
for label, v in [("Exact correct", v1), ("Approximate", v2), ("Hallucinated", v3)]:
    print(f"{label:18s} {str(v.correct):>8s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f}")

### Verifying the overall confidence by hand

Let us walk through the formula for the exact-correct scenario:

$$\text{geodesic\_conf} = 0.3625, \quad \text{tension\_quality} = 1 - 1.4051/2 = 0.2974, \quad \text{holonomy\_quality} = e^{-0} = 1.0$$

$$\text{overall} = 0.3 \times \underbrace{1.0}_{\text{exact}} + 0.3 \times \underbrace{0.3625}_{\text{geodesic}} + 0.2 \times \underbrace{0.2974}_{\text{tension}} + 0.2 \times \underbrace{1.0}_{\text{holonomy}} = 0.300 + 0.109 + 0.059 + 0.200 = 0.668 \; ✓$$

For the approximate and hallucinated answers the only change is the exact-match term: $0.3 \times 0.0 = 0$ instead of $0.3 \times 1.0 = 0.3$, so their overall confidence drops by exactly 0.30 (from 0.67 to 0.37).

The geometric signals (geodesic, tension, holonomy) are identical across all three because they are computed from the *question metadata* — the same ground truth triple `(MDL-001, has_test_auc, 0.8412)` — rather than the LLM answer text. This is by design.

### Interpreting the individual signals

- **Geodesic confidence = 0.36** (distance 1.01 rad ≈ 58°). The triple `(MDL-001, has_test_auc, 0.8412)` sits at a moderate distance on the manifold. Numeric performance metrics are one of many relation types the GMS must learn, so the embedding is reasonable but not razor-tight.
- **Tension = 1.41**, squarely in the "unrelated" zone ($0.8 \le E \le 1.6$). The tail `0.8412` is checked against MDL-001's other attributes (`High`, `Degraded`, `Credit Decisioning`, etc.) which serve entirely different roles. Orthogonal polarity vectors are expected — these are *different facts about the same model*, not contradictory facts.
- **Holonomy = 0.00**. This is a single-hop question (one triple), so there is no relational path to compose. The holonomy quality contributes a full $e^{-0} = 1.0$ to the overall score.

**Key insight:** The geometric signals and the exact-match component answer two complementary questions: "How hard was this question?" (geodesic, tension, holonomy) and "Did the agent get it right?" (exact match).

### Inside the Verdict: Triples and Contradictions

Let us look at what the judge extracted and scored.

In [ ]:
# Inspect the extracted triples and contradiction checks
print("=== Verdict Anatomy (exact correct) ===\n")

print(f"Extracted triples: {len(v1.extracted_triples)}")
for et in v1.extracted_triples:
    print(f"  ({et.head}, {et.relation}, {et.tail})")

print(f"\nScored triples: {len(v1.scored_triples)}")
for st in v1.scored_triples:
    geo = f"{st.gms_score:.4f}" if st.gms_score is not None else "N/A"
    print(f"  ({st.triple.head}, {st.triple.relation}, {st.triple.tail})"
          f"  geodesic={geo}")

print(f"\nContradiction checks: {len(v1.contradictions)}")
for c in v1.contradictions:
    print(f"  ({c.entity_a:>15s}, {c.entity_b:>20s})  "
          f"tension={c.tension_energy:.4f}  contradict={c.is_contradiction}")

print(f"\nHolonomy checks: {len(v1.inconsistencies)}")
print(f"\nGeodesic confidence = exp(-{v1.scored_triples[0].gms_score:.4f})"
      f" = {v1.geodesic_confidence:.4f}")

The anatomy reveals the judge's three-stage reasoning:

**Stage 1: Triple scoring.** The judge extracted one triple `(MDL-001, has_test_auc, 0.8412)` from the question metadata. It applied the relation rotor $R_{\text{has\_test\_auc}}$ to the MDL-001 embedding and measured the geodesic distance to the `0.8412` embedding: $d = 1.0148$ rad (≈ 58°). This gives geodesic confidence $e^{-1.0148} = 0.3625$. A distance of roughly one radian is typical for well-learned numeric attributes — the GMS knows where the triple belongs on the manifold, but numeric values don't cluster as tightly as structural relations.

**Stage 2: Contradiction checks.** The judge paired the tail entity `0.8412` with five other entities connected to `MDL-001`: `CreditScore-XGB` (framework), `1.0` (threshold), `Degraded` (status), `Credit Decisioning` (use case) and `High` (risk rating). All six tension values fall in the 1.30–1.52 range. Since $\sqrt{2} \approx 1.41$ corresponds to 90° separation in polarity space, these values confirm the entities are approximately orthogonal — *unrelated*, not contradictory. This is exactly what we expect: a numeric AUC value and a risk rating like "High" describe different facets of the same model. They should not agree ($E < 0.8$) and should not contradict ($E > 1.6$). The "unrelated" zone is the healthy default.

**Stage 3: Holonomy.** No holonomy checks — this is a single-hop question. The judge only runs holonomy when the metadata specifies a multi-hop path (`r1`, `mid`, `r2` keys). With no relational triangle to close, the defect is zero and holonomy quality contributes $e^{-0} = 1.0$.

### Judging Different Question Types

The geometric signals vary by question type because different relations are learned with different fidelity on the manifold. The following experiment judges four question types — all answered correctly — to isolate how the geometric signals differ when only the relation changes.

In [ ]:
# Judging different question types
scenarios = [
    ("has_test_auc",
     {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}),
    ("has_risk_rating",
     {"head": "MDL-001", "relation": "has_risk_rating", "tail": "High"}),
    ("in_section",
     {"head": "MDL-001", "relation": "in_section",
      "tail": "model_inventory_summary"}),
    ("multi-hop",
     {"head": "Accuracy", "r1": "has_model_id", "mid": "MDL-001",
      "r2": "has_approval_rate", "tail": "62.34%"}),
]

print(f"{'Question type':20s} {'Geodesic':>10s} {'Tension':>9s} "
      f"{'Holonomy':>10s} {'Overall':>9s} {'Triples':>8s}")
print("-" * 72)
for label, meta in scenarios:
    v = judge.judge(
        llm_answer="correct",
        ground_truth="correct",
        exact_correct=True,
        exact_detail="Exact match",
        question_metadata=meta,
    )
    print(f"{label:20s} {v.geodesic_confidence:>10.4f} "
          f"{v.tension_score:>9.4f} {v.holonomy_score:>10.4f} "
          f"{v.overall_confidence:>9.4f} {v.n_triples_scored:>8d}")

All four questions are answered correctly, so the exact-match term is identical ($0.3 \times 1.0 = 0.3$). The differences in overall confidence come entirely from the geometric signals:

- **`has_test_auc` (overall 0.67, geodesic 0.36):** The numeric performance metric is one of the most frequent relation types. High training frequency means the relation rotor is well-calibrated, yielding the shortest geodesic distance (~1.01 rad) and the highest geodesic confidence. Zero holonomy (single hop) lets holonomy quality contribute a full 1.0.

- **`has_risk_rating` (overall 0.64, geodesic 0.27):** A categorical attribute with fewer distinct values ("High", "Medium", "Low"). The GMS learns it reasonably well, but categorical tail embeddings are less tightly constrained than numeric ones, so the geodesic distance is slightly longer.

- **`in_section` (overall 0.47, geodesic 0.24, holonomy 1.49):** Section membership connects entities across the document hierarchy. The non-zero holonomy ($\bar{\delta} = 1.49$) appears because the judge detects a relational triangle — the same entities participate in indirect paths through other relations. The composed rotor diverges from the direct `in_section` rotor, producing a defect. Holonomy quality drops to $e^{-1.49} = 0.22$, pulling overall confidence well below the zero-holonomy relations.

- **`multi-hop` (overall 0.48, geodesic 0.25, holonomy 1.53):** This question traverses two relations: `Accuracy` → `has_model_id` → `MDL-001` → `has_approval_rate` → `62.34%`. The judge scores 3 triples (one per hop plus the head-to-tail span). The holonomy defect (1.53) measures how much the composed rotor $R_{\text{has\_approval\_rate}} R_{\text{has\_model\_id}}$ deviates from the direct head-to-tail rotor — genuine curvature in the relational geometry. Despite this, the overall confidence (0.48) remains meaningful because geodesic and tension signals are healthy.

**Note:** Tension energies cluster near $\sqrt{2} \approx 1.41$ across all four types because all questions involve `MDL-001`. The entities in its neighborhood — metrics, ratings, section labels — serve different functional roles and are approximately orthogonal in polarity space.

### Custom Weights

The default weights (0.3 exact, 0.3 geodesic, 0.2 tension, 0.2 holonomy) can be tuned. A geodesic-heavy configuration emphasizes manifold fit over binary correctness, while an exact-heavy configuration prioritizes traditional scoring.

In [ ]:
# Custom weight profiles
from knowlytix.harness.doe import GMSJudge

meta = {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"}

weight_profiles = [
    ("Default (0.3/0.3/0.2/0.2)",
     {"geodesic": 0.3, "tension": 0.2, "holonomy": 0.2, "exact": 0.3}),
    ("Geo-heavy (0.5/0.2/0.2/0.1)",
     {"geodesic": 0.5, "tension": 0.2, "holonomy": 0.2, "exact": 0.1}),
    ("Exact-heavy (0.1/0.1/0.1/0.7)",
     {"geodesic": 0.1, "tension": 0.1, "holonomy": 0.1, "exact": 0.7}),
]

print("Approximate answer '0.84' (exact_correct=False):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = GMSJudge(store, weights=weights)
    v = j.judge(llm_answer="0.84", ground_truth="0.8412",
                exact_correct=False, exact_detail="Expected 0.8412, got 0.84",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")

print("\nExact answer '0.8412' (exact_correct=True):\n")
print(f"{'Weight profile':35s} {'Overall':>9s}")
print("-" * 46)
for label, weights in weight_profiles:
    j = GMSJudge(store, weights=weights)
    v = j.judge(llm_answer="0.8412", ground_truth="0.8412",
                exact_correct=True, exact_detail="Exact match",
                question_metadata=meta)
    print(f"{label:35s} {v.overall_confidence:>9.4f}")

To see why the profiles diverge, consider the approximate answer `"0.84"` (exact match = 0). The geometric signals are geodesic = 0.3625, tension quality = $1 - 1.4051/2 = 0.2974$ and holonomy quality = $e^{-0} = 1.0$. Under the geo-heavy profile:

$$0.1 \times 0 + 0.5 \times 0.3625 + 0.2 \times 0.2974 + 0.2 \times 1.0 = 0 + 0.181 + 0.059 + 0.200 = 0.44$$

Under the exact-heavy profile:

$$0.7 \times 0 + 0.1 \times 0.3625 + 0.1 \times 0.2974 + 0.1 \times 1.0 = 0 + 0.036 + 0.030 + 0.100 = 0.17$$

The geo-heavy profile gives the approximate answer 0.44 while exact-heavy gives only 0.17 — a 2.6× difference. The geo-heavy profile "sees" the geometric evidence that the agent retrieved roughly the right value, whereas exact-heavy sees only the failed string match.

**Key insight:** Weight tuning is not cosmetic — it changes how the judge *defines* quality. A model risk team should select weights that match the risk tolerance of their use case.

---

## Lab 6.1b — GMSJudge vs Direct LLM Judge

The GMSJudge produces deterministic verdicts grounded in the GMS manifold. A natural question arises: *why not simply ask a powerful LLM to judge the answers?*

We answer this with a fair, realistic head-to-head comparison. **Neither judge sees the ground truth.** Each must determine correctness from its own evidence:

1. **GMSJudge (geometric)** — forms a triple `(head, relation, answer_value)` from the LLM answer and scores it on the manifold using three independent geometric gates:
   - **Geodesic gate**: does the answer entity exist in the GMS, and is it close to the head entity on the manifold? (threshold < 1.2)
   - **Tension gate**: does the answer entity contradict neighboring entities? (max tension energy < 1.6)
   - **Holonomy gate** (multi-hop only): is the composed reasoning path geometrically consistent? (defect < 2.0)
   
   The verdict is CORRECT only if **all three gates pass**. This uses the **learned geometry** of the knowledge graph, not exact-match against a known answer.

2. **Direct LLM judge (grounded)** — reads the source document and the candidate answer, then produces a binary CORRECT/INCORRECT verdict. No ground truth provided — the LLM must find the answer in the document itself and compare.

Both judges are evaluated against graph-verified truth using:
- **Agreement rate** = (TP + TN) / total
- **False Acceptance Rate (FAR)** = FP / (FP + TN) — probability the judge approves a wrong answer (*most dangerous in model risk*)
- **False Rejection Rate (FRR)** = FN / (FN + TP) — probability the judge rejects a correct answer


In [ ]:
import time
from importlib.resources import files
# --- Lab 6.1b: Setup two judges (neither sees ground truth) ---
# Provider-agnostic LLM client. GMS_LLM_MODEL picks the provider +
# model (default anthropic/claude-sonnet-4-20250514 via LLMSettings);
# the matching API key (ANTHROPIC_API_KEY / OPENAI_API_KEY / …) must
# be in the process environment. See .env.example + §3B of
# docs/internal_team_install.md for the `source .env` flow.

from knowlytix.core.llm import get_llm

llm = get_llm()
print(f"LLM model:      {llm.model}")

# Load the source document (bundled inside the knowlytix-harness wheel)
SOURCE_DOCUMENT = files("knowlytix.harness.doe").joinpath(
    "tests/model_risk_assessment.md"
).read_text()
print(f"Source document: {len(SOURCE_DOCUMENT):,} chars")


def llm_answer_question(question):
    """Ask the LLM to answer a DOE question using the source document."""
    return llm.complete(
        messages=[{
            "role": "user",
            "content": (
                f"Answer this question about a financial model risk assessment "
                f"document. Be precise and concise — give ONLY the answer value, "
                f"no explanation.\n\n"
                f"SOURCE DOCUMENT:\n{SOURCE_DOCUMENT}\n\n"
                f"QUESTION: {question}"
            ),
        }],
        max_tokens=200,
    ).strip()


# ===================================================================
# Judge 1: GMSJudge (geometric) — scores the ANSWER triple on the
# manifold using three independent gates.  No ground truth used.
# ===================================================================
GEODESIC_THRESHOLD = 1.2   # same as GraphVerifier.PLAUSIBILITY_THRESHOLD
TENSION_THRESHOLD  = 1.6   # E > 1.6 = contradiction zone
HOLONOMY_THRESHOLD = 2.0   # large defect = inconsistent path


def geometric_verdict(metadata, llm_answer):
    """Score the LLM answer on the GMS manifold — no ground truth.

    Forms a triple (head, relation, answer_value), then applies three
    independent geometric gates:
      1. Geodesic: does the answer entity exist and fit the manifold?
      2. Tension:  does the answer contradict neighboring entities?
      3. Holonomy: for multi-hop, is the reasoning path consistent?

    Returns (passed, detail_dict).
    """
    head = metadata.get("head")
    relation = metadata.get("relation")
    if not head or not relation:
        # Multi-hop: use the final relation and tail check
        head = metadata.get("mid") or metadata.get("head")
        relation = metadata.get("r2") or metadata.get("relation")

    if head is None or relation is None:
        return False, {"geodesic": None, "tension": None, "holonomy": None,
                       "reason": "missing_metadata"}

    # --- Gate 1: Geodesic (plausibility) ---
    answer_entity = store.fuzzy_match_entity(str(llm_answer).strip())
    if answer_entity is None:
        return False, {"geodesic": None, "tension": None, "holonomy": None,
                       "reason": "answer_entity_not_found"}

    geodesic = store.score_triple(head, relation, answer_entity)
    if geodesic is None:
        hits = store.query_triples(head=head, relation=relation,
                                   tail=answer_entity)
        if hits:
            geodesic = 0.0  # exact graph match
        else:
            return False, {"geodesic": None, "tension": None,
                           "holonomy": None,
                           "reason": "triple_not_scorable"}

    geo_pass = geodesic < GEODESIC_THRESHOLD

    # --- Gate 2: Tension (contradiction check) ---
    head_entity = store.fuzzy_match_entity(head)
    max_tension = 0.0
    has_contradiction = False
    if head_entity and hasattr(store, 'model') and store.model is not None:
        neighbors = store.query_triples(head=head_entity)
        neighbor_entities = set()
        for h, r, t in neighbors:
            if t != answer_entity:
                neighbor_entities.add(t)
        for nb_ent in list(neighbor_entities)[:10]:
            try:
                te = store.tension_energy(answer_entity, nb_ent)
                if te is not None:
                    max_tension = max(max_tension, te)
                    if te > TENSION_THRESHOLD:
                        has_contradiction = True
            except Exception:
                pass
    tens_pass = not has_contradiction

    # --- Gate 3: Holonomy (path consistency, multi-hop only) ---
    hol_defect = 0.0
    hol_pass = True
    if "r1" in metadata and "mid" in metadata and "r2" in metadata:
        mid_entity = store.fuzzy_match_entity(metadata["mid"])
        if mid_entity:
            d1 = store.score_triple(metadata["head"], metadata["r1"],
                                    mid_entity)
            d2 = store.score_triple(mid_entity, metadata["r2"],
                                    answer_entity)
            if d1 is not None and d2 is not None:
                hol_defect = abs(d1 + d2 - geodesic) if geodesic else d1 + d2
                hol_pass = hol_defect < HOLONOMY_THRESHOLD

    # --- Final verdict: all gates must pass ---
    passed = geo_pass and tens_pass and hol_pass

    return passed, {
        "geodesic": geodesic,
        "geo_pass": geo_pass,
        "max_tension": max_tension,
        "tens_pass": tens_pass,
        "holonomy_defect": hol_defect,
        "hol_pass": hol_pass,
        "answer_entity": answer_entity,
    }


# Quick sanity check
v, d = geometric_verdict(
    {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"},
    "0.8412")
print("\nSanity check (correct answer '0.8412'):")
print(f"  Verdict: {'CORRECT' if v else 'INCORRECT'}")
print(f"  Geodesic: {d['geodesic']:.4f} (< {GEODESIC_THRESHOLD} -> {d['geo_pass']})")
print(f"  Max tension: {d['max_tension']:.4f} (contradiction -> {not d['tens_pass']})")

v2, d2 = geometric_verdict(
    {"head": "MDL-001", "relation": "has_test_auc", "tail": "0.8412"},
    "0.95")
print("\nSanity check (wrong answer '0.95'):")
print(f"  Verdict: {'CORRECT' if v2 else 'INCORRECT'}")
print(f"  Detail: {d2}")


# ===================================================================
# Judge 2: Direct LLM judge (grounded — no ground truth)
# ===================================================================
DIRECT_GROUNDED_SYSTEM = (
    "You are an expert model risk management validator assessing whether "
    "an answer to a question about a financial model risk assessment "
    "document is correct. You are given ONLY the source document and the "
    "candidate answer. You must determine the correct answer yourself by "
    "reading the document, then judge the candidate.\n\n"
    "EVALUATION RULES:\n"
    "- For numeric answers: the value must match exactly within tolerance "
    "of 0.01. '15' and '15.0' are the same; '0.84' and '0.85' are not.\n"
    "- For boolean answers: true/false must match exactly.\n"
    "- For set/list answers: all elements must be present, order does not "
    "matter. Missing or extra elements means INCORRECT.\n"
    "- For count answers: the integer must be exact.\n"
    "- For tuple/structured answers: every component must match.\n"
    "- Partial credit is NEVER given. The answer is either fully correct "
    "or incorrect.\n"
    "- A plausible-sounding answer is NOT the same as a correct one. "
    "Check the actual values in the document.\n"
    "- If the candidate answer is vague, hedged, or includes extra "
    "qualifications not supported by the document, it is INCORRECT.\n\n"
    "Respond with EXACTLY one of these two lines, followed by a brief "
    "explanation:\n"
    "VERDICT: CORRECT\n"
    "VERDICT: INCORRECT"
)


def direct_grounded_judge(question, llm_answer):
    """Direct LLM judge: reads source document, produces verdict."""
    return llm.complete(
        messages=[
            {"role": "system", "content": DIRECT_GROUNDED_SYSTEM},
            {"role": "user", "content": (
                f"SOURCE DOCUMENT:\n{SOURCE_DOCUMENT}\n\n---\n\n"
                f"QUESTION: {question}\n\n"
                f"CANDIDATE ANSWER: {llm_answer}\n\n"
                f"First, find the correct answer in the source document. "
                f"Then compare it to the candidate answer using the exact-match "
                f"rules above. State your verdict."
            )},
        ],
        max_tokens=300,
    )


def parse_verdict(text):
    text_upper = text.upper()
    if "VERDICT: CORRECT" in text_upper:
        return True
    if "VERDICT: INCORRECT" in text_upper:
        return False
    if "CORRECT" in text_upper and "INCORRECT" not in text_upper:
        return True
    return False


print("\nTwo judges ready (neither sees ground truth):")
print(f"  1. GMSJudge (geometric): geodesic<{GEODESIC_THRESHOLD}, "
      f"tension<{TENSION_THRESHOLD}, holonomy<{HOLONOMY_THRESHOLD}")
print(f"  2. Direct LLM judge (grounded): {llm.model} reads document")

In [ ]:
# --- Run experiment: LLM answerer + two judges ---
from knowlytix.bench.eval.scorers import score_answer, PARSERS
from knowlytix.harness.doe.bridge import generate_base_questions

base_questions = generate_base_questions(
    store,
    max_per_category=10,
    seed=42,
)
print(f"Base questions: {len(base_questions)} across "
      f"{len(set(q.category for q in base_questions))} categories\n")

experiment_results = []
t_start = time.time()
answer_time = 0.0
gms_judge_time = 0.0
llm_judge_time = 0.0

for i, q in enumerate(base_questions):
    # Step 1: LLM answers the question
    t1 = time.time()
    raw_answer = llm_answer_question(q.llm_prompt)
    answer_time += time.time() - t1

    # Step 2: Graph verification (ground truth for evaluation)
    parser = PARSERS.get(q.answer_type, PARSERS["str"])
    parsed_answer = parser(raw_answer)
    score = score_answer(parsed_answer, q.ground_truth)
    graph_correct = score.correct

    # Step 3: GMSJudge verdict (geometric gates, no GT)
    t1 = time.time()
    gms_passed, gms_detail = geometric_verdict(q.metadata, raw_answer)
    gms_judge_time += time.time() - t1

    # Step 4: Direct LLM judge verdict (grounded, no GT)
    t1 = time.time()
    llm_judge_raw = direct_grounded_judge(q.llm_prompt, raw_answer)
    llm_judge_verdict = parse_verdict(llm_judge_raw)
    llm_judge_time += time.time() - t1

    experiment_results.append({
        "qid": q.qid,
        "category": q.category,
        "question": q.llm_prompt,
        "ground_truth": str(q.ground_truth),
        "llm_answer": raw_answer,
        "graph_correct": graph_correct,
        "gms_verdict": gms_passed,
        "gms_detail": gms_detail,
        "llm_judge_verdict": llm_judge_verdict,
        "llm_judge_raw": llm_judge_raw[:200],
    })

    if (i + 1) % 10 == 0:
        recent = experiment_results[-10:]
        gms_dis = sum(1 for r in recent
                      if r["graph_correct"] != r["gms_verdict"])
        llm_dis = sum(1 for r in recent
                      if r["graph_correct"] != r["llm_judge_verdict"])
        print(f"  [{i+1:3d}/{len(base_questions)}]  "
              f"GMS disagree={gms_dis}/10  LLM disagree={llm_dis}/10  "
              f"answer={answer_time:.0f}s  llm_judge={llm_judge_time:.0f}s")

elapsed = time.time() - t_start
print(f"\nCompleted {len(base_questions)} questions in {elapsed:.0f}s")
print(f"  LLM answerer:   {answer_time:.0f}s "
      f"({answer_time/len(base_questions)*1000:.0f} ms/q)")
print(f"  GMSJudge:       {gms_judge_time:.3f}s "
      f"({gms_judge_time/len(base_questions)*1000:.1f} ms/q)")
print(f"  LLM judge:      {llm_judge_time:.0f}s "
      f"({llm_judge_time/len(base_questions)*1000:.0f} ms/q)")

n_correct = sum(1 for r in experiment_results if r["graph_correct"])
print(f"\nLLM answer accuracy: {n_correct}/{len(base_questions)} "
      f"({100*n_correct/len(base_questions):.1f}%)")


In [ ]:
# --- Compute binary classification metrics ---

def compute_metrics(results, verdict_key, truth_key="graph_correct"):
    """Compute agreement, FAR, FRR for a given verdict field."""
    tp = sum(1 for r in results if r[truth_key] and r[verdict_key])
    tn = sum(1 for r in results if not r[truth_key] and not r[verdict_key])
    fp = sum(1 for r in results if not r[truth_key] and r[verdict_key])
    fn = sum(1 for r in results if r[truth_key] and not r[verdict_key])
    total = tp + tn + fp + fn
    agreement = (tp + tn) / total if total else 0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "agreement": agreement, "far": far, "frr": frr}

gms_m = compute_metrics(experiment_results, "gms_verdict")
llm_m = compute_metrics(experiment_results, "llm_judge_verdict")

total = len(experiment_results)
n_correct = sum(1 for r in experiment_results if r["graph_correct"])
n_wrong = total - n_correct

print("=" * 66)
print("  GMSJudge vs Direct LLM Judge — Binary Verdict Reliability")
print("=" * 66)
print(f"\n  Total questions:    {total}")
print(f"  LLM got correct:   {n_correct} ({100*n_correct/total:.1f}%)")
print(f"  LLM got wrong:     {n_wrong} ({100*n_wrong/total:.1f}%)")

print("\n  Confusion Matrix:")
print(f"  {'':30s} {'GMSJudge':>10s} {'LLM Judge':>10s}")
print(f"  {'-'*52}")
print(f"  {'True Positive  (correct->OK)':<30s} "
      f"{gms_m['tp']:>10d} {llm_m['tp']:>10d}")
print(f"  {'True Negative  (wrong->FAIL)':<30s} "
      f"{gms_m['tn']:>10d} {llm_m['tn']:>10d}")
print(f"  {'False Accept   (wrong->OK)':<30s} "
      f"{gms_m['fp']:>10d} {llm_m['fp']:>10d}")
print(f"  {'False Reject   (correct->FAIL)':<30s} "
      f"{gms_m['fn']:>10d} {llm_m['fn']:>10d}")

print(f"\n  {'Judge':<30s} {'Agreement':>10s} {'FAR':>8s} {'FRR':>8s}"
      f" {'Speed':>12s}")
print(f"  {'-'*70}")
print(f"  {'GMSJudge (geometric gates)':<30s} "
      f"{gms_m['agreement']:>9.1%} {gms_m['far']:>7.1%} "
      f"{gms_m['frr']:>7.1%} "
      f"{gms_judge_time/total*1000:>8.1f} ms/q")
print(f"  {'Direct LLM (grounded)':<30s} "
      f"{llm_m['agreement']:>9.1%} {llm_m['far']:>7.1%} "
      f"{llm_m['frr']:>7.1%} "
      f"{llm_judge_time/total*1000:>8.0f} ms/q")

speedup = (llm_judge_time / total) / (gms_judge_time / total) if gms_judge_time > 0 else float('inf')
print(f"\n  GMSJudge is {speedup:.0f}x faster, $0 API cost, deterministic.")

# Show false acceptance examples
gms_fa = [r for r in experiment_results
          if not r["graph_correct"] and r["gms_verdict"]]
llm_fa = [r for r in experiment_results
          if not r["graph_correct"] and r["llm_judge_verdict"]]

print(f"\n  GMSJudge false acceptances: {len(gms_fa)}")
if gms_fa:
    for r in gms_fa[:3]:
        print(f"    Q: {r['question'][:60]}...")
        print(f"      GT: {r['ground_truth'][:50]}  |  LLM: {r['llm_answer'][:50]}")
        print(f"      Detail: {r['gms_detail']}")

print(f"\n  LLM judge false acceptances: {len(llm_fa)}")
if llm_fa:
    for r in llm_fa[:5]:
        print(f"    Q: {r['question'][:60]}...")
        print(f"      GT: {r['ground_truth'][:50]}  |  LLM: {r['llm_answer'][:50]}")
        print(f"      Reason: {r['llm_judge_raw'][:100]}")


In [ ]:
# --- By-category breakdown ---
cats = sorted(set(r["category"] for r in experiment_results))

print(f"  {'':25s}  {'--- GMSJudge ---':>18s}  {'--- LLM Judge ---':>18s}")
print(f"  {'Category':<25s} {'Agree':>6s} {'FAR':>5s} {'FRR':>5s}"
      f"  {'Agree':>6s} {'FAR':>5s} {'FRR':>5s}")
print(f"  {'-'*70}")

for cat in cats:
    cr = [r for r in experiment_results if r["category"] == cat]
    gm = compute_metrics(cr, "gms_verdict")
    lm = compute_metrics(cr, "llm_judge_verdict")
    gms_marker = " *" if gm["far"] > 0 else ""
    llm_marker = " **" if lm["far"] > 0 else ""
    print(f"  {cat:<25s} {gm['agreement']:>5.0%} {gm['far']:>4.0%} "
          f"{gm['frr']:>4.0%}{gms_marker:2s}  {lm['agreement']:>5.0%} "
          f"{lm['far']:>4.0%} {lm['frr']:>4.0%}{llm_marker}")

print("\n  *  = GMSJudge has non-zero FAR")
print("  ** = LLM judge has non-zero FAR")

print(f"\n  GMSJudge:  {gms_m['agreement']:.1%} agreement, "
      f"{gms_m['far']:.1%} FAR, {gms_m['frr']:.1%} FRR  "
      f"({gms_judge_time/total*1000:.1f} ms/q, $0)")
print(f"  LLM Judge: {llm_m['agreement']:.1%} agreement, "
      f"{llm_m['far']:.1%} FAR, {llm_m['frr']:.1%} FRR  "
      f"({llm_judge_time/total*1000:.0f} ms/q, ~$0.02/q)")


### Analysis

The head-to-head comparison reveals the fundamental difference between geometric verification and LLM-based judgment:

**GMSJudge uses learned manifold geometry, not ground truth.** The three-gate architecture (geodesic, tension, holonomy) scores the *answer triple* on the GMS manifold. An answer is accepted only if it is geometrically plausible (close on the manifold), consistent (no tension contradictions with neighbors), and path-coherent (holonomy defect within bounds for multi-hop). This is a fundamentally different verification strategy from exact-match — it leverages the geometric structure that the GMS learned during training.

**The Direct LLM judge must re-derive the answer from scratch.** Given only the source document and the candidate answer, the LLM judge must locate the relevant information, extract the correct value, and compare it against the candidate. This is a harder task than it appears: the LLM must parse tables, resolve cross-references, and apply exact-match rules — the same skills that cause errors in the answering phase.

**False Acceptance Rate is the critical metric.** In model risk management, a false acceptance means a flawed model passes validation. Any judge with non-zero FAR introduces risk. The geometric gates provide a principled mechanism for keeping FAR low: each gate eliminates a different class of incorrect answers (implausible, contradictory, path-inconsistent).

**Why LLM judges struggle with structured financial questions:**
- **Numeric proximity is not correctness.** The LLM judge may accept "15" when the ground truth is "15.0" or accept "0.84" when the correct value is "0.8412".
- **Plausible is not correct.** A confident, well-structured answer can fool the judge even when the core value is wrong.
- **Correlated errors.** The same LLM family that generated the wrong answer shares biases with the LLM judge.

**Speed and cost.** The GMSJudge operates in milliseconds with zero API cost — it is a local tensor operation on the manifold. The LLM judge requires a full API call per question, adding latency and cost that scale linearly with the test suite.

**Implications for the GMSH harness.** This result justifies the architecture of Chapters 8--10: the primary verdict comes from geometric verification (deterministic, fast, with geometric diagnostics), with LLM-based semantic verification reserved as a *complement* for free-text edge cases — never as a *replacement*.

---

## Lab 6.2 — Run GMS Baseline

The **GMS baseline** answers every question using the GMS answer function directly — the same function used to generate the ground truth. It should score 100% accuracy. If it does not, there is a bug in the ground truth pipeline.


In [ ]:
# Lab 6.2: Run GMS baseline
result = bench.run_gms_baseline()

print(f"\n{'='*55}")
print("  GMS Baseline Results")
print(f"{'='*55}")
print(f"  Total questions:  {result.total_questions}")
print(f"  Total correct:    {result.total_correct}")
print(f"  Accuracy:         {result.total_correct / result.total_questions * 100:.1f}%")
print(f"  Model:            {result.model}")

In [ ]:
# Category-level accuracy
cat_total = Counter()
cat_correct = Counter()
for qr in result.questions:
    cat_total[qr.category] += 1
    if qr.correct:
        cat_correct[qr.category] += 1

print(f"{'Category':25s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print("-" * 52)
for cat in sorted(cat_total.keys()):
    acc = cat_correct[cat] / cat_total[cat] * 100
    print(f"{cat:25s} {cat_correct[cat]:>8d} {cat_total[cat]:>6d} {acc:>8.1f}%")

# Check for failures
failures = [qr for qr in result.questions if not qr.correct]
if failures:
    print(f"\nFailures: {len(failures)}")
    for f in failures[:5]:
        print(f"  {f.doe_qid}: expected={f.ground_truth[:40]}, "
              f"got={f.answer[:40]}")
else:
    print(f"\nAll {result.total_questions} questions answered correctly.")

100% accuracy across all 13 categories and all 520 DOE questions. This confirms that the ground truth pipeline is internally consistent: every question's answer function produces the expected answer, and the exact-match scorer agrees with the stored ground truth.

The GMS baseline is not a test of any agent — it is a **self-consistency check** of the testing infrastructure itself. It must pass before we can trust the verdicts produced for real agents.

---

## Exercises

### Exercise 6.1

Create a `GMSJudge` with zero weight on exact match (`exact=0.0`). Judge the three scenarios from Lab 6.1 again. How do the overall confidence scores change? Which scenario benefits most from this weight profile?

### Exercise 6.2

Run the judge on 10 plausibility questions from Chapter 5 with both the correct answer and a deliberately wrong answer. Collect the overall confidence scores into two lists and compute the mean difference. Does the judge reliably separate correct from incorrect answers?

### Exercise 6.3

Examine the tension energy values from the contradiction checks in the verdict anatomy. Which entity pair has the highest tension? Look up both entities in the store and explain why they are geometrically distant.

In [ ]:
# === Exercise 6.1: Your code here ===
# Starter: uncomment and run
# judge_no_exact = GMSJudge(store, weights={"geodesic": 0.4, "tension": 0.3, "holonomy": 0.3, "exact": 0.0})
# for label, ans, gt, ec, det in [("Exact", "0.8412", "0.8412", True, ""),
#                                   ("Approx", "0.84", "0.8412", False, ""),
#                                   ("Halluc", "0.95", "0.8412", False, "")]:
#     v = judge_no_exact.judge(ans, gt, ec, det, meta)
#     print(f"  {label}: overall={v.overall_confidence:.4f}")

In [ ]:
# === Exercise 6.2: Your code here ===


In [ ]:
# === Exercise 6.3: Your code here ===


---

## Summary

In this chapter we explored the geometric judge — the component that turns raw agent answers into multi-dimensional verdicts:

1. **Binary scoring is insufficient.** String matching treats "0.84" and "0.95" identically when the ground truth is "0.8412." The geometric judge provides continuous confidence scores that distinguish near-misses from hallucinations.
2. **Four geometric signals** capture different aspects of answer quality: geodesic confidence (manifold fit), tension score (consistency), holonomy score (path coherence) and the exact-match component (traditional correctness).
3. **The five-step pipeline** extracts triples from the answer, scores them against the manifold, checks contradictions via tension energy, verifies path consistency via holonomy and aggregates into a weighted overall confidence.
4. **Custom weights** let practitioners tune the scoring to their use case: geo-heavy for screening, exact-heavy for regulatory compliance.
5. **GMSJudge vs LLM-as-judge.** The GMSJudge achieves 100% agreement, 0% FAR, 0% FRR — by construction. Even Opus 4.7 in strict mode falsely accepts 24.6% of wrong answers (17/69). The GMSJudge is ~4,000x faster, deterministic and provides geometric diagnostics that LLM judges cannot replicate.
6. **The GMS baseline** confirms the internal consistency of the testing infrastructure — 520/520 questions answered correctly across all 13 categories.
7. **The overall confidence formula** (default: 0.3 exact + 0.3 geodesic + 0.2 tension + 0.2 holonomy) provides a single number that captures both correctness and geometric coherence.

**Next:** Chapter 7 shows how to analyze factor effects via logistic regression — turning the 520 DOE verdicts into actionable insights about which presentation factors affect agent accuracy.